# A/B Channel Definition Test for fRG Kagome

这个 notebook 的目标很单纯：

**不改 flow 物理，只比较 diagnosis-stage 的 longitudinal particle-hole channel 定义。**

它对比两套版本：

- **旧版**：`frg_flow.py` + `channels.py`
- **补丁版**：`frg_flow_channelfix.py` + `channels_channelfix.py`

核心问题是：

> `ph_charge_longitudinal` / `ph_spin_longitudinal` 的定义本身，是否已经把 FM / PI 的 sector 搞歪了？

---

## 这个 notebook 会做什么

1. 用**同一套** `patchsets + bare_gamma + solver_kwargs` 跑旧版和新版 flow  
2. 检查 flow state 是否一致  
   - `pp_corr`
   - `phd_corr`
   - `phc_corr`
3. 在给定 `Q` 上比较 diagnosis kernels：
   - `ph_charge_longitudinal`
   - `ph_spin_longitudinal`
   - legacy 对照版本
4. 输出 mixed raw blocks 的范数：
   - `uu_to_dd`
   - `dd_to_uu`

---

## 判读逻辑

### 如果出现：
- flow state 基本完全一致
- 但 corrected longitudinal kernels 和 legacy kernels 明显不同
- 且 mixed blocks 非零

那说明：

> **问题已经在 channel definition 层出现了**  
> 不是单纯 form-factor diagnosis 才出问题。

### 如果出现：
- corrected 和 legacy 几乎一样
- mixed blocks 也几乎为 0

那说明：

> **channel 这条怀疑不成立**  
> 下一步应重点去查 `form_factor.py` / `kagome_order_diagnosis.py`


In [1]:
import sys
from pathlib import Path

DATA_DIR = Path("/mnt/data")
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

print("Using data dir:", DATA_DIR)
print("Check files:")
for name in [
    "frg_flow.py",
    "channels.py",
    "frg_flow_channelfix.py",
    "channels_channelfix.py",
    "test_channel_ab_compare.py",
]:
    p = DATA_DIR / name
    print(f"  {name}: {'OK' if p.exists() else 'MISSING'}")


Using data dir: \mnt\data
Check files:
  frg_flow.py: MISSING
  channels.py: MISSING
  frg_flow_channelfix.py: MISSING
  channels_channelfix.py: MISSING
  test_channel_ab_compare.py: MISSING


## 1. 导入测试工具

这里直接调用已经准备好的 A/B 测试脚本。

In [2]:
from test_channel_ab_compare import run_channel_ab_test, print_channel_ab_report
from frg_flow import BareVertexFromInteraction


In [3]:
import time
import numpy as np
import matplotlib.pyplot as plt

from noninteracting import KagomeNagaosa
from patching import FSPatcher, plot_patchset
from interaction import BareExtendedHubbard
from kagome_order_diagnosis import KagomeOrderDiagnoser

from frg_flow import BareVertexFromInteraction
from frg_flow import FRGFlowSolver
# -----------------------------
# physical parameters
# -----------------------------
t = 1.0
phi = 0.0          # real hopping only
filling = 5.0 / 12.0
U = 0.0
V = 10
Npatch = 12

model = KagomeNagaosa(
    parameters={"t": t, "phi": phi},
    spin=True,
    B=None,
)

print("Model built.")
print("b1 =", model.b1)
print("b2 =", model.b2)
t0 = time.perf_counter()
# mu = model.EF_from_filling(filling, N=180, tol=1e-5, maxiter=80)
mu=0
t1 = time.perf_counter()

print(f"mu(EF) for filling={filling:.6f}: {mu:.10f}")
print(f"EF solve time: {t1 - t0:.3f} s")
# -----------------------------
# build patch sets for up / dn
# -----------------------------
t0 = time.perf_counter()

patcher_up = FSPatcher(
    model,
    band_index=1,                 # middle dispersive band near 5/12; adjust to your convention if needed
    mu=mu,
    orbital_slice=slice(0, 3),
    Npatch=Npatch,
    grid_size=320,
    gauge_fix="parallel_transport",
    close_loop_gauge=True,
    verbose=False,
)

patcher_dn = FSPatcher(
    model,
    band_index=1,
    mu=mu,
    orbital_slice=slice(3, 6),
    Npatch=Npatch,
    grid_size=320,
    gauge_fix="parallel_transport",
    close_loop_gauge=True,
    verbose=False,
)

patchset_up = patcher_up.build()
patchset_dn = patcher_dn.build()

patchsets = {"up": patchset_up, "dn": patchset_dn}

t1 = time.perf_counter()

print(f"Patch construction time: {t1 - t0:.3f} s")
print("Npatch(up) =", patchset_up.Npatch)
print("Npatch(dn) =", patchset_dn.Npatch)
print("gauge(up)  =", patchset_up.gauge_method)
print("gauge(dn)  =", patchset_dn.gauge_method)
print("mu used    =", patchset_up.mu)

Model built.
b1 = [0.         3.62759873]
b2 = [3.14159265 1.81379936]
mu(EF) for filling=0.416667: 0.0000000000
EF solve time: 0.000 s
Patch construction time: 26.529 s
Npatch(up) = 12
Npatch(dn) = 12
gauge(up)  = parallel_transport
gauge(dn)  = parallel_transport
mu used    = 0.0


In [4]:
t0 = time.perf_counter()

interaction = BareExtendedHubbard.from_kagome_model(model, U=U, V=V)
# bare_gamma = BareVertexFromInteraction(patchsets=patchsets, interaction=hint)

t1 = time.perf_counter()

print(f"Bare interaction / bare gamma build time: {t1 - t0:.3f} s")
# print("Norb =", hint.Norb)

Bare interaction / bare gamma build time: 0.000 s


## 2. 在这里准备你自己的 `patchsets` 和 `interaction`

这个 notebook **不会替你重新构造模型**，因为你自己的 pipeline 里已经有：

- `interaction`
- `patchsets`

你只需要确保当前 notebook 内已经存在这两个变量。

### 变量要求

- `interaction`：通常是 `BareExtendedHubbard` 或兼容对象
- `patchsets`：字典，通常形如
  ```python
  {
      "up": patchset_up,
      "dn": patchset_dn,
  }
  ```

如果你想，也可以先运行你已有 pipeline 中构造模型/patch 的 cell，再继续执行下面。

In [5]:
# 运行这个 cell 只是做检查，不会创建对象
required = ["interaction", "patchsets"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "当前 notebook 里还没有这些变量: "
        + ", ".join(missing)
        + "\n请先运行你已有 pipeline 中构造 interaction / patchsets 的 cells。"
    )

print("interaction:", type(interaction))
print("patchsets keys:", list(patchsets.keys()))
for k, ps in patchsets.items():
    npatch = getattr(ps, "Npatch", None)
    print(f"  spin={k!r}, Npatch={npatch}")


interaction: <class 'interaction.BareExtendedHubbard'>
patchsets keys: ['up', 'dn']
  spin='up', Npatch=12
  spin='dn', Npatch=12


## 3. 构造 bare gamma

这里使用你原来的 `interaction + patchsets` 来构造 bare vertex accessor。

In [6]:
bare_gamma = BareVertexFromInteraction(interaction, patchsets)
print("bare_gamma ready:", bare_gamma)


bare_gamma ready: <frg_flow.BareVertexFromInteraction object at 0x000002926A9EA4A0>


## 4. 设置 flow 参数

这里先给一套**适中**参数，方便你快速比较。  
之后你可以直接改成你真实项目里常用的参数。

In [7]:
solver_kwargs = dict(
    T_start=0.5,
    T_stop=0.02,
    n_steps=12,
    nfreq=128,
    diagnose_every=1,
    # 你也可以按需要加：
    # channel_divergence_threshold=1e3,
    # eigenvalue_threshold=1e2,
    # max_relative_update=0.15,
)

solver_kwargs


{'T_start': 0.5,
 'T_stop': 0.02,
 'n_steps': 12,
 'nfreq': 128,
 'diagnose_every': 1}

## 5. 运行 A/B 测试

这一步会分别运行：

- 旧版 `frg_flow`
- 新版 `frg_flow_channelfix`

并输出对比报告。

In [8]:
report = run_channel_ab_test(
    patchsets=patchsets,
    bare_gamma=bare_gamma,
    old_flow_module="frg_flow",
    new_flow_module="frg_flow_channelfix",
    solver_kwargs=solver_kwargs,
)
print_channel_ab_report(report)


{
  "state_comparison": {
    "pp_corr_max_diff": 0.0,
    "phd_corr_max_diff": 0.0,
    "phc_corr_max_diff": 0.0,
    "final_channel_norm_old": 0.0,
    "final_channel_norm_new": 0.0,
    "n_history_old": 1,
    "n_history_new": 1
  },
  "kernel_report": [
    {
      "Q": [
        0.0,
        0.0
      ],
      "old_available": [
        "ph_charge_longitudinal",
        "ph_spin_longitudinal",
        "phc_dd",
        "phc_uu",
        "phd_dd",
        "phd_uu",
        "pp_du_to_du",
        "pp_du_to_ud",
        "pp_singlet_sz0",
        "pp_triplet_dd",
        "pp_triplet_sz0",
        "pp_triplet_uu",
        "pp_ud_to_du",
        "pp_ud_to_ud"
      ],
      "new_available": [
        "ph_charge_longitudinal",
        "ph_charge_longitudinal_legacy",
        "ph_spin_longitudinal",
        "ph_spin_longitudinal_legacy",
        "phc_dd",
        "phc_uu",
        "phd_dd",
        "phd_dd_to_dd",
        "phd_dd_to_uu",
        "phd_uu",
        "phd_uu_to_dd",
        "

## 6. 查看关键数值

这里把最关键的定量结果单独拎出来。

In [11]:
print("=== Flow state difference ===")
for key in ["pp_corr_max_diff", "phd_corr_max_diff", "phc_corr_max_diff"]:
    print(f"{key}: {report.get(key)}")

print("\n=== Mixed raw block norms ===")
for key, val in report.get("mixed_block_norms", {}).items():
    print(f"{key}: {val}")

print("\n=== Compared kernels ===")
for key in sorted(report.get("kernel_comparison", {}).keys()):
    info = report["kernel_comparison"][key]
    print(f"\n[{key}]")
    for kk, vv in info.items():
        print(f"  {kk}: {vv}")


=== Flow state difference ===
pp_corr_max_diff: None
phd_corr_max_diff: None
phc_corr_max_diff: None

=== Mixed raw block norms ===

=== Compared kernels ===


## 7. 更清楚地看 kernel labels

如果你主要关心：

- corrected 的 `ph_charge_longitudinal`
- corrected 的 `ph_spin_longitudinal`
- 以及它们与 legacy 的差别

可以只看这些。

In [10]:
focus = [
    "new::ph_charge_longitudinal",
    "new::ph_spin_longitudinal",
    "new::ph_charge_longitudinal_legacy",
    "new::ph_spin_longitudinal_legacy",
    "old::ph_charge_longitudinal",
    "old::ph_spin_longitudinal",
]

for key in focus:
    if key not in report.get("kernel_comparison", {}):
        print(f"{key}: NOT FOUND")
        continue
    info = report["kernel_comparison"][key]
    print(f"\n=== {key} ===")
    for kk in [
        "eig_abs_max",
        "hermitian_residual",
        "leading_label",
        "leading_score",
        "diagnosis_available",
    ]:
        print(f"{kk}: {info.get(kk)}")


new::ph_charge_longitudinal: NOT FOUND
new::ph_spin_longitudinal: NOT FOUND
new::ph_charge_longitudinal_legacy: NOT FOUND
new::ph_spin_longitudinal_legacy: NOT FOUND
old::ph_charge_longitudinal: NOT FOUND
old::ph_spin_longitudinal: NOT FOUND


## 8. 如何判读

### 情况 A：channel definition 确实有问题
如果你看到：

- `pp_corr_max_diff`, `phd_corr_max_diff`, `phc_corr_max_diff` 都接近 0
- 但 corrected longitudinal kernels 和 legacy kernels 差很多
- mixed blocks `uu_to_dd`, `dd_to_uu` 明显非零

那说明：

> **flow 本身没变，但 diagnosis-stage 的 channel 组合方式改变了结果**  
> 也就是 **channel definition 层已经有问题**

---

### 情况 B：channel 怀疑不成立
如果你看到：

- flow state 差异接近 0
- corrected 和 legacy 的 kernels 几乎一样
- mixed blocks 也几乎为 0

那说明：

> **问题不在 channel definition**  
> 下一步应该重点转向：
> - `form_factor.py`
> - `kagome_order_diagnosis.py`

---

## 9. 建议你接下来贴给我的内容

你跑完以后，把下面这些结果贴给我就够了：

1. `pp_corr_max_diff / phd_corr_max_diff / phc_corr_max_diff`
2. `mixed_block_norms`
3. 下面 6 个 kernel 的输出：
   - `new::ph_charge_longitudinal`
   - `new::ph_spin_longitudinal`
   - `new::ph_charge_longitudinal_legacy`
   - `new::ph_spin_longitudinal_legacy`
   - `old::ph_charge_longitudinal`
   - `old::ph_spin_longitudinal`

这样我们就能很快判断下一步到底该改哪里。